# Evaluate the Optuna-tuned chronological decoder

Given the quarter history, predict **which team performs the next recorded action, which action it is, and its location**, or a neutral boundary. This notebook reads saved evaluation reports. It does not train, load model weights or rescore matches.

The decoder completed a 60-trial search, three-seed finalist selection and three fresh refits. Seq2Event was paused before its search finished. A separate evaluation-order amendment scored the frozen decoder first. The combined experiment remains unfinished.

Candidates fit on **2023–2024**, select on **2025**, refit on **2023–2025** and are evaluated on **2026**. The 2026 data is an already-observed project benchmark. Full protocol and limitations are in the [evaluation note](../docs/decoder-evaluation.md).

## 1. Load the compact report

The committed JSON/CSV files support a complete walkthrough without raw archives or model weights. The manifest records the three frozen refits before test data was loaded. Paths shown below are relative to the repository.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
root = next(p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").is_file() and (p / "data/sources.json").is_file())
report = root / "reports/examples/event-decoder-optuna-v1"

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

summary = read_json(report / "summary.json")
manifest = read_json(report / "frozen_decoder.json")
selection = read_json(report / "selection.json")
metrics = pd.read_csv(report / "metrics.csv")
aggregates = pd.read_csv(report / "metrics-summary.csv")
assert summary["status"] == "complete" and summary["scope"] == "decoder_only"
assert set(manifest["models"]) == set(map(str, summary["seeds"]))
print("Report:", report.relative_to(root).as_posix())
print("Evaluation frozen (UTC):", manifest["created_utc"])
print("Decoder evaluation:", summary["status"])
print("Source combined experiment:", summary["source_experiment_status"])
print("Amendment:", summary["amendment"])

Report: reports/examples/event-decoder-optuna-v1
Evaluation frozen (UTC): 2026-10-26T00:58:33.918033+00:00
Decoder evaluation: complete
Source combined experiment: incomplete
Amendment: Evaluate the completed decoder study before the Seq2Event study finishes. Decoder configuration, validation selection, refit duration, three seeds and weights remain fixed. The source combined experiment remains unfinished.


## 2. Inspect validation selection

Pruned trials count towards the fixed 60-trial budget. The top three completed configurations were each fitted with three seeds. Mean best validation objective selects the configuration. The median best epoch fixes refit duration for every seed. Trial numbers below are zero-based.

The objective is five times joint cross-entropy plus normalised location RMSE. Class weights are one. Test results do not enter this selection.

In [2]:
trials = pd.read_csv(report / "trials.csv")
display(trials.groupby("state").size().rename("trials").to_frame())
display(pd.DataFrame([{k: c[k] for k in ["trial", "mean_validation_loss", "std_validation_loss", "refit_epochs"]} for c in selection["candidates"]]))
winner = selection["winner"]
display(pd.Series(winner["config"], name="selected value").to_frame())
display(pd.DataFrame(winner["seeds"])[["seed", "best_epoch", "epochs_completed", "stopping_reason", "validation_loss"]])
print("Fresh refit epochs for every seed:", winner["refit_epochs"])

,trials
state,
COMPLETE,36
PRUNED,24


,trial,mean_validation_loss,std_validation_loss,refit_epochs
0,38,5.531943,0.004300,56
1,46,5.539129,0.002846,57
2,56,5.542971,0.001921,56


,selected value
mode,joint
width,64
embedding_dim,8
club_embedding_dim,4
heads,4
layers,4
ff_multiplier,2
dense_dim,128
dropout,0.261275
learning_rate,0.0019


,seed,best_epoch,epochs_completed,stopping_reason,validation_loss
0,2026,50,60,early_stopping,5.531118
1,2027,57,60,epoch_limit,5.536596
2,2028,56,60,epoch_limit,5.528116


Fresh refit epochs for every seed: 56


The selected width and embeddings are at the smallest offered values, while layer count and output hidden dimension are at the largest. Two finalist seeds reached the 60-epoch limit and selected epochs near that limit. This is a bounded search, not evidence that further optimisation cannot help.

## 3. Check the evaluation population

Joint classification has 25 classes: twelve actions for each side, plus one neutral boundary. Location error is calculated only where coordinate supervision is valid. Quality filters exclude uncertain transitions from scoring while retaining observations in the input history. Histories reset at each quarter.

In [3]:
audit = read_json(report / "test_audit.json")
test = metrics.loc[metrics["split"].eq("test")].copy()
display(pd.Series({k: audit[k] for k in ["quarters", "tokens", "ball_actions", "boundaries", "gaps", "ineligible_tokens"]}, name="raw quarter audit").to_frame())
display(test[["model", "seed", "examples", "location_examples", "boundary_rate"]])
print("Coordinate frame:", audit["coordinate_frame"])
assert test["examples"].nunique() == test["location_examples"].nunique() == 1
assert set(test.loc[test.model.eq("transformer"), "seed"]) == set(summary["seeds"])

,raw quarter audit
quarters,869
tokens,357223
ball_actions,324461
boundaries,31345
gaps,1417
ineligible_tokens,20434


,model,seed,examples,location_examples,boundary_rate
4,transformer,2026.0,333883,304298,0.088609
5,transition,NaN,333883,304298,0.088609
6,transformer,2027.0,333883,304298,0.088609
7,transformer,2028.0,333883,304298,0.088609


Coordinate frame: Home attacking direction within each quarter; dimensionless venue-normalised coordinates.


## 4. Compare all seeds with the transition baseline

The transition baseline uses the current team/action code to estimate the next code and an average training displacement to estimate location. It was fitted on the refit seasons and is scored once.

Higher accuracy and macro F1 are better. Lower log loss and location RMSE are better. Accuracy covers the full team/action/boundary target. Macro F1 weights classes equally, making weak rare-action performance more visible. Locations use dimensionless venue-normalised coordinates.

The aggregate reports the mean and **sample standard deviation across three independently refitted seeds**. This is not a three-model ensemble and the SD is not uncertainty across matches.

In [4]:
wanted = ["joint_accuracy", "joint_log_loss", "joint_macro_f1", "location_rmse_norm"]
display(test[["model", "seed", *wanted]].round(6))
test_aggregates = aggregates.loc[aggregates["split"].eq("test") & aggregates.metric.isin(wanted)]
display(test_aggregates[["model", "metric", "mean", "std", "seeds"]].round(6))

previous_report = root / "reports/examples/event-comparison-v1"
assert read_json(previous_report / "test_audit.json") == audit, "Previous benchmark population differs"
previous_metrics = pd.read_csv(previous_report / "metrics.csv")
previous = previous_metrics.loc[previous_metrics["mode"].eq("joint") & previous_metrics["split"].eq("test") & previous_metrics.model.eq("transformer")].iloc[0]
means = test_aggregates.loc[test_aggregates.model.eq("transformer")].set_index("metric")["mean"]
baseline = test.loc[test.model.eq("transition")].iloc[0]
comparison = pd.DataFrame([baseline[wanted], previous[wanted], means[wanted]], index=["Transition baseline", "Earlier decoder (one seed)", "Optuna decoder (three-seed mean)"])
display(comparison.astype(float).round(6))
print(f"Accuracy change vs earlier decoder: {100 * (means['joint_accuracy'] - previous['joint_accuracy']):.3f} percentage points")
for key in ["joint_log_loss", "location_rmse_norm"]:
    print(f"{key} reduction vs earlier decoder: {100 * (1 - means[key] / previous[key]):.2f}%")

,model,seed,joint_accuracy,joint_log_loss,joint_macro_f1,location_rmse_norm
4,transformer,2026.0,0.636699,1.056941,0.336806,0.124165
5,transition,NaN,0.578559,1.234567,0.236605,0.199207
6,transformer,2027.0,0.636427,1.056369,0.326689,0.126719
7,transformer,2028.0,0.635297,1.059711,0.337843,0.123747


,model,metric,mean,std,seeds
51,transformer,location_rmse_norm,0.124877,0.001609,3
54,transformer,joint_accuracy,0.636141,0.000743,3
55,transformer,joint_log_loss,1.057673,0.001787,3
56,transformer,joint_macro_f1,0.333780,0.006162,3
75,transition,location_rmse_norm,0.199207,0.000000,1
78,transition,joint_accuracy,0.578559,0.000000,1
79,transition,joint_log_loss,1.234567,0.000000,1
80,transition,joint_macro_f1,0.236605,0.000000,1


,joint_accuracy,joint_log_loss,joint_macro_f1,location_rmse_norm
Transition baseline,0.578559,1.234567,0.236605,0.199207
Earlier decoder (one seed),0.630832,1.079540,0.315859,0.135143
Optuna decoder (three-seed mean),0.636141,1.057673,0.333780,0.124877


Accuracy change vs earlier decoder: 0.531 percentage points
joint_log_loss reduction vs earlier decoder: 2.03%
location_rmse_norm reduction vs earlier decoder: 7.60%


The tuned decoder improves all four reported metrics over the earlier decoder and transition baseline on this benchmark. The earlier search had a smaller budget and only one seed. These differences do not isolate one architectural change, establish statistical significance or show that the model can support a particular defensive tactic.

## 5. Inspect confusion matrices

Rows are actual classes and columns are predicted classes. The complete joint matrix below uses **seed 2026**, the predeclared first seed. It was not selected for its test performance. Reports contain every seed's matrices and the baseline matrices.

The two-by-two team matrices are marginal diagnostics restricted to actual ordinary-action targets, with boundary probability removed and the remaining probabilities renormalised. They should not be mistaken for full joint accuracy.

In [5]:
SEED = 2026
assert SEED in summary["seeds"]
stem = f"test-transformer-seed-{SEED}"
joint = pd.read_csv(report / f"{stem}-joint-confusion.csv", index_col=0)
assert joint.to_numpy().sum() == int(test.iloc[0]["examples"])
with pd.option_context("display.max_rows", 30, "display.max_columns", 30):
    display(joint)
for name in ["transition", f"transformer-seed-{SEED}"]:
    print(name)
    display(pd.read_csv(report / f"test-{name}-team_given_action-confusion.csv", index_col=0))
print("Full matrices: test-transformer-seed-<seed>-<task>-confusion.csv")

,home:bounce,home:free_kick,home:gather,home:ground_kick,home:handball,home:hard_ball_get,home:kick,home:kick_in,home:kick_in_play_on,home:loose_ball_get,home:mark,home:receive,away:bounce,away:free_kick,away:gather,away:ground_kick,away:handball,away:hard_ball_get,away:kick,away:kick_in,away:kick_in_play_on,away:loose_ball_get,away:mark,away:receive,boundary
actual,,,,,,,,,,,,,,,,,,,,,,,,,
home:bounce,63,2,0,0,610,0,1079,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
home:free_kick,1,436,31,4,231,0,399,0,71,739,611,275,0,16,30,0,500,0,217,0,4,337,343,156,884
home:gather,0,40,286,4,10,0,5,0,9,1213,1755,233,0,50,97,0,1,0,0,0,9,683,213,34,955
home:ground_kick,0,6,49,28,10,0,4,0,2,333,183,179,0,5,22,0,8,0,1,0,2,169,67,95,250
home:handball,9,17,1,1,23264,0,8388,0,0,0,0,17,0,0,0,1,0,0,0,0,0,0,0,0,19
home:hard_ball_get,0,34,48,4,26,0,3,0,15,1229,597,253,0,37,82,1,24,0,8,0,12,593,464,444,1025
home:kick,19,54,0,0,11088,0,31806,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
home:kick_in,0,0,0,0,1,0,1,0,87,2,0,2,0,0,0,0,0,0,0,0,0,0,10,1,96
home:kick_in_play_on,0,6,0,0,7,0,7,0,920,27,3,27,0,0,0,0,0,0,0,0,0,3,42,5,1037


transition


,home,away
actual,,
home,140024,14271
away,25202,124801


transformer-seed-2026


,home,away
actual,,
home,138141,16154
away,19765,130238


Full matrices: test-transformer-seed-<seed>-<task>-confusion.csv


## 6. Examine per-class performance across seeds

Mean recall and F1 below give each seed equal weight. Support is unchanged across runs. Zero recall means that no examples of that class were predicted correctly, even when overall accuracy is high. The full reports also retain precision and each seed's individual results.

In [6]:
classes = pd.concat([pd.read_csv(report / f"test-transformer-seed-{seed}-per-class.csv").assign(seed=seed) for seed in summary["seeds"]], ignore_index=True)
per_class = classes.loc[classes.task.eq("joint")].groupby("label", sort=False).agg(support=("support", "first"), precision_mean=("precision", "mean"), recall_mean=("recall", "mean"), f1_mean=("f1", "mean"), f1_std=("f1", "std"))
with pd.option_context("display.max_rows", 30):
    display(per_class.sort_values("recall_mean").round(4))
conditional = aggregates.loc[aggregates["split"].eq("test") & aggregates.model.eq("transformer") & aggregates.metric.isin(["team_given_action_accuracy", "action_given_action_accuracy"])]
display(conditional[["metric", "mean", "std", "seeds"]].round(6))

,support,precision_mean,recall_mean,f1_mean,f1_std
label,,,,,
home:kick_in,200,0.0000,0.0000,0.0000,0.0000
home:hard_ball_get,4899,0.0000,0.0000,0.0000,0.0000
away:kick_in,195,0.0000,0.0000,0.0000,0.0000
away:hard_ball_get,4692,0.0000,0.0000,0.0000,0.0000
away:ground_kick,1539,0.2854,0.0058,0.0114,0.0045
home:ground_kick,1413,0.4540,0.0160,0.0308,0.0070
home:bounce,1754,0.7274,0.0319,0.0610,0.0195
away:bounce,1542,0.7195,0.0376,0.0714,0.0120
away:gather,5570,0.2511,0.0468,0.0788,0.0102


,metric,mean,std,seeds
60,team_given_action_accuracy,0.881973,0.000055,3
66,action_given_action_accuracy,0.684554,0.000571,3


## 7. Interpret the results

The decoder can estimate likely next events from a quarter history with club context. The improved location and probability scores support further investigation of team-level patterns. Opponent-specific validation and a reporting layer are still needed before these predictions become an opposition analysis product.

- 2026 is an already-observed benchmark, not an untouched holdout.
- The decoder was evaluated before Seq2Event finished. Its remaining search must stay frozen when resumed.
- All three decoder seeds are reported. No seed is selected using the test results.
- Rare actions remain weak, and quality filters favour cleaner passages.
- Location forecasts are point estimates. Forecast accuracy does not establish which tactical response would improve an outcome.

See the [evaluation note](../docs/decoder-evaluation.md), [full report files](../reports/examples/event-decoder-optuna-v1/) and [formulation walkthrough](13_event_formulations.ipynb).